In [2]:
pip -q install "jedi>=0.16,<1.0"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 22.1 MB/s eta 0:00:00


In [3]:
!pip -q install -U llama-index llama-index-readers-file llama-index-embeddings-huggingface \
  llama-index-vector-stores-faiss faiss-cpu pypdf transformers accelerate sentencepiece

In [4]:
from google.colab import files
import os, shutil

os.makedirs("/content/pdfs", exist_ok=True)
uploaded = files.upload()

for name in uploaded.keys():
    shutil.move(name, f"/content/pdfs/{name}")

print("PDFs:", os.listdir("/content/pdfs"))

Saving WHO_Pharmacological treatment of hypertension in adults.pdf to WHO_Pharmacological treatment of hypertension in adults.pdf
PDFs: ['WHO_Pharmacological treatment of hypertension in adults.pdf']


In [5]:
import faiss
from llama_index.core.readers import SimpleDirectoryReader
from llama_index.core import VectorStoreIndex, Settings
from llama_index.core.node_parser import SentenceSplitter
from llama_index.vector_stores.faiss import FaissVectorStore
from llama_index.embeddings.huggingface import HuggingFaceEmbedding

# Embeddings (open-source)
Settings.embed_model = HuggingFaceEmbedding(model_name="sentence-transformers/all-MiniLM-L6-v2")

# Load PDFs
docs = SimpleDirectoryReader("/content/pdfs").load_data()

# Auto chunking (no manual chunking)
splitter = SentenceSplitter(chunk_size=512, chunk_overlap=80)
nodes = splitter.get_nodes_from_documents(docs)

# FAISS store
dim = 384  # all-MiniLM-L6-v2 embedding dim
faiss_index = faiss.IndexFlatIP(dim)
vector_store = FaissVectorStore(faiss_index=faiss_index)

# Build index
index = VectorStoreIndex(nodes, vector_store=vector_store)

print("Loaded docs:", len(docs))
print("Built nodes:", len(nodes))

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Loaded docs: 61
Built nodes: 116


In [6]:
retriever = index.as_retriever(similarity_top_k=5)

def show_hits(query: str):
    hits = retriever.retrieve(query)
    print("QUERY:", query)
    print("-" * 80)
    for i, h in enumerate(hits, 1):
        src = h.node.metadata.get("file_name") or h.node.metadata.get("source") or "unknown"
        print(f"[{i}] score={h.score:.4f}  source={src}")
        print(h.node.text[:400].replace("\n"," ") + " ...")
        print()

show_hits("What is the main contribution of this paper?")

QUERY: What is the main contribution of this paper?
--------------------------------------------------------------------------------
[1] score=0.2412  source=WHO_Pharmacological treatment of hypertension in adults.pdf
 ...

[2] score=0.2409  source=WHO_Pharmacological treatment of hypertension in adults.pdf
Very low We have very little confidence in the effect estimate. (The true effect is likely to be  substantially different from the estimate of effect.) The strength of the recommendations reflects the degree of confidence of the GDG that the desirable  effects (e.g. beneficial health outcomes) of the recommendations outweigh the undesirable effects  (e.g. adverse effects). The strength of recommen ...

[3] score=0.2228  source=WHO_Pharmacological treatment of hypertension in adults.pdf
 ...

[4] score=0.2141  source=WHO_Pharmacological treatment of hypertension in adults.pdf
 ...

[5] score=0.2104  source=WHO_Pharmacological treatment of hypertension in adults.pdf
GUIDELINE FOR THE 

In [13]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import torch

MODEL_NAME = "google/flan-t5-large"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME).to("cuda")

MAX_INPUT_TOKENS = 1024
RESERVED_FOR_INSTRUCTIONS = 120   # leave space for question + prompt text
CTX_BUDGET = MAX_INPUT_TOKENS - RESERVED_FOR_INSTRUCTIONS

def pack_context_to_budget(chunks, tokenizer, budget_tokens: int):
    """Greedy pack chunks until token budget is reached."""
    packed = []
    used = 0
    for ch in chunks:
        # count tokens for this chunk (+ a separator)
        ch_tokens = len(tokenizer.encode(ch, add_special_tokens=False))
        sep_tokens = 2
        if used + ch_tokens + sep_tokens > budget_tokens:
            # if nothing fits yet, hard-truncate the first chunk
            if not packed:
                ids = tokenizer.encode(ch, add_special_tokens=False)[:budget_tokens]
                return tokenizer.decode(ids)
            break
        packed.append(ch)
        used += ch_tokens + sep_tokens
    return "\n\n---\n\n".join(packed)

def rag_answer_safe(question: str, hits, max_new_tokens=180):
    # hits = retriever.retrieve(question)
    raw_chunks = [h.node.text for h in hits]   # already ranked by relevance
    context = pack_context_to_budget(raw_chunks, tokenizer, CTX_BUDGET)

    prompt = (
        "Answer the question using ONLY the context. "
        "If the context is insufficient, say you don't know.\n\n"
        f"Question: {question}\n\n"
        f"Context:\n{context}\n\n"
        "Answer:"
    )

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=MAX_INPUT_TOKENS
    ).to(model.device)

    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=max_new_tokens)

    return tokenizer.decode(out[0], skip_special_tokens=True)

In [15]:
hits = retriever.retrieve("Recommenadations for Hypertension")
print(rag_answer_safe("Recommendations for hypertension", hits))

not enough information


In [19]:
def debug_retrieve(q, k=5):
    hits = index.as_retriever(similarity_top_k=k).retrieve(q)
    for i,h in enumerate(hits,1):
        src = h.node.metadata.get("file_name","?")
        print(f"[{i}] score={h.score:.4f} source={src}")
        print(h.node.text[:300].replace("\n"," "), "...\n")
    return hits

hits = debug_retrieve("Recommendations for hypertension", k=10)

[1] score=0.6486 source=WHO_Pharmacological treatment of hypertension in adults.pdf
An overview of systematic reviews of the evidence was used to build  summary of findings tables according to the Grading of Recommendations, Assessment, Development  and Evaluations (GRADE) approach. The Guideline Development Group developed recommendations,  considering the certainty of the evidenc ...

[2] score=0.6356 source=WHO_Pharmacological treatment of hypertension in adults.pdf
Executive summary More people die each year from cardiovascular diseases than from any other cause. Over three  quarters of heart disease and stroke-related deaths occur in low-income and middle-income countries.  Hypertension – or elevated blood pressure – is a serious medical condition that signif ...

[3] score=0.6339 source=WHO_Pharmacological treatment of hypertension in adults.pdf
(50) Chow CK, Teo KK, Rangarajan S, Islam S, Gupta R, Avezum A, et al. Prevalence, awareness,  treatment, and control of hypertension in

In [20]:
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from llama_index.core import Settings

Settings.embed_model = HuggingFaceEmbedding(
    model_name="sentence-transformers/all-MiniLM-L6-v2",
    normalize=True
)

In [21]:
Settings.embed_model = HuggingFaceEmbedding(
    model_name="pritamdeka/S-PubMedBert-MS-MARCO",
    normalize=True
)

modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/123 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/666 [00:00<?, ?B/s]

...

model.safetensors:   0%|          | 0.00/438M [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [22]:
from llama_index.core.node_parser import SentenceSplitter
splitter = SentenceSplitter(chunk_size=900, chunk_overlap=150)
nodes = splitter.get_nodes_from_documents(docs)
# rebuild index after this

In [23]:
hits = retriever.retrieve(
    "WHO guideline recommendations: pharmacological treatment of hypertension in adults "
    "initiation threshold first-line drugs target blood pressure follow-up"
)
debug_retrieve("WHO guideline recommendations: pharmacological treatment of hypertension in adults initiation threshold first-line drugs target blood pressure follow-up", k=10)

[1] score=0.7421 source=WHO_Pharmacological treatment of hypertension in adults.pdf
An overview of systematic reviews of the evidence was used to build  summary of findings tables according to the Grading of Recommendations, Assessment, Development  and Evaluations (GRADE) approach. The Guideline Development Group developed recommendations,  considering the certainty of the evidenc ...

[2] score=0.7401 source=WHO_Pharmacological treatment of hypertension in adults.pdf
GUIDELINE FOR THE PHARMACOLOGICAL TREATMENT OF HYPERTENSION IN ADULTS WHO suggests pharmacological antihypertensive treatment of individuals without  cardiovascular disease but with high cardiovascular risk, diabetes mellitus, or chronic kidney  disease, and systolic blood pressure of 130–139 mmHg.  ...

[3] score=0.7371 source=WHO_Pharmacological treatment of hypertension in adults.pdf
Executive summary More people die each year from cardiovascular diseases than from any other cause. Over three  quarters of heart diseas

[NodeWithScore(node=TextNode(id_='531717f8-c670-407f-8e76-1bdc72dddce4', embedding=None, metadata={'page_label': 'vii', 'file_name': 'WHO_Pharmacological treatment of hypertension in adults.pdf', 'file_path': '/content/pdfs/WHO_Pharmacological treatment of hypertension in adults.pdf', 'file_type': 'application/pdf', 'file_size': 843539, 'creation_date': '2026-01-25', 'last_modified_date': '2026-01-25'}, excluded_embed_metadata_keys=['file_name', 'file_type', 'file_size', 'creation_date', 'last_modified_date', 'last_accessed_date'], excluded_llm_metadata_keys=['file_name', 'file_type', 'file_size', 'creation_date', 'last_modified_date', 'last_accessed_date'], relationships={<NodeRelationship.SOURCE: '1'>: RelatedNodeInfo(node_id='9978e91e-fa9e-44f8-b812-4d43ef88d67b', node_type='4', metadata={'page_label': 'vii', 'file_name': 'WHO_Pharmacological treatment of hypertension in adults.pdf', 'file_path': '/content/pdfs/WHO_Pharmacological treatment of hypertension in adults.pdf', 'file_type

In [24]:
KEY_PHRASES = [
    "recommend", "recommendation", "strong recommendation", "conditional recommendation",
    "target blood pressure", "initiation", "threshold", "first-line", "treatment goal",
    "<140/90", "<130", "mmhg", "algorithm", "implementation"
]

def boost_recommendation_chunks(hits):
    boosted = []
    for h in hits:
        t = h.node.text.lower()
        bonus = sum(1 for k in KEY_PHRASES if k in t)
        boosted.append((h, bonus))
    boosted.sort(key=lambda x: (x[1], x[0].score), reverse=True)
    return [h for h,_ in boosted]

hits = boost_recommendation_chunks(retriever.retrieve("Recommendations for hypertension"))
for i,h in enumerate(hits[:10],1):
    print(i, h.score, h.node.metadata.get("file_name","?"))
    print(h.node.text[:200].replace("\n"," "), "\n")

1 0.6486097090246725 WHO_Pharmacological treatment of hypertension in adults.pdf
An overview of systematic reviews of the evidence was used to build  summary of findings tables according to the Grading of Recommendations, Assessment, Development  and Evaluations (GRADE) approach.  

2 0.6356150657521384 WHO_Pharmacological treatment of hypertension in adults.pdf
Executive summary More people die each year from cardiovascular diseases than from any other cause. Over three  quarters of heart disease and stroke-related deaths occur in low-income and middle-incom 

3 0.623854632123388 WHO_Pharmacological treatment of hypertension in adults.pdf
GUIDELINE FOR THE PHARMACOLOGICAL TREATMENT OF HYPERTENSION IN ADULTS This guideline will replace the guidance in the modules Evidence-based protocols and Risk-based CVD  management of the HEARTS tech 

4 0.6339332452161751 WHO_Pharmacological treatment of hypertension in adults.pdf
(50) Chow CK, Teo KK, Rangarajan S, Islam S, Gupta R, Avezum A, et a

In [25]:
import re

def looks_like_citations(text: str) -> bool:
    t = text.strip()
    if len(t) < 80:
        return True
    # lots of (12) Author... doi: patterns
    if re.search(r"\bdoi:\b", t.lower()) and re.search(r"\(\d+\)", t):
        return True
    if t.count(";") > 8 and t.count(".") < 3:
        return True
    return False

def rag_answer_safe_filtered(question: str, hits, max_new_tokens=180):
    # keep only useful chunks
    useful = [h for h in hits if not looks_like_citations(h.node.text)]
    if not useful:
        useful = hits  # fallback

    useful = boost_recommendation_chunks(useful)

    raw_chunks = [h.node.text for h in useful]
    context = pack_context_to_budget(raw_chunks, tokenizer, CTX_BUDGET)

    prompt = (
        "Answer the question using ONLY the context. "
        "If the context is insufficient, say you don't know.\n\n"
        f"Question: {question}\n\nContext:\n{context}\n\nAnswer:"
    )

    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=MAX_INPUT_TOKENS).to(model.device)
    out = model.generate(**inputs, max_new_tokens=max_new_tokens)
    return tokenizer.decode(out[0], skip_special_tokens=True)

In [26]:
hits = retriever.retrieve("Recommendations for hypertension")
print(rag_answer_safe_filtered("Give the WHO guideline recommendations for pharmacological treatment of hypertension in adults. Include thresholds, targets, and first-line drug options.", hits))

WHO recommends initiation of pharmacological antihypertensive treatment of individuals with a confirmed diagnosis of hypertension and systolic blood pressure of 140 mmHg or diastolic blood pressure of 90 mmHg


Second RAG


In [32]:
import os, urllib.request

os.makedirs("/content/pdfs_llm", exist_ok=True)

url = "https://aclanthology.org/2025.naacl-long.328.pdf"
out = "/content/pdfs_llm/naacl2025_decoding_speculative_decoding.pdf"

urllib.request.urlretrieve(url, out)
print("Saved:", out)

Saved: /content/pdfs_llm/naacl2025_decoding_speculative_decoding.pdf


In [35]:
import faiss
from llama_index.core.readers import SimpleDirectoryReader
from llama_index.core import VectorStoreIndex, Settings
from llama_index.core.node_parser import SentenceSplitter
from llama_index.vector_stores.faiss import FaissVectorStore
from llama_index.embeddings.huggingface import HuggingFaceEmbedding

# IMPORTANT: keep the same embedding settings as RAG #1 for now
# (If you already set Settings.embed_model earlier, you can omit this block.)
Settings.embed_model = HuggingFaceEmbedding(
    model_name="sentence-transformers/all-MiniLM-L6-v2",
    normalize=True
)

# Load PDF(s) for RAG #2
docs2 = SimpleDirectoryReader("/content/pdfs_llm").load_data()

# Use a separate splitter instance (so you don't accidentally change splitter for RAG #1)
splitter2 = SentenceSplitter(chunk_size=512, chunk_overlap=80)
nodes2 = splitter2.get_nodes_from_documents(docs2)

# Build FAISS index (same dim = 384 for all-MiniLM-L6-v2)
faiss_index2 = faiss.IndexFlatIP(384)
vector_store2 = FaissVectorStore(faiss_index=faiss_index2)

index2 = VectorStoreIndex(nodes2, vector_store=vector_store2)
retriever2 = index2.as_retriever(similarity_top_k=8)

print("RAG #2 docs:", len(docs2))
print("RAG #2 nodes:", len(nodes2))

RAG #2 docs: 14
RAG #2 nodes: 43


In [36]:
def debug_retrieve2(q, k=10):
    hits = index2.as_retriever(similarity_top_k=k).retrieve(q)
    for i, h in enumerate(hits, 1):
        src = h.node.metadata.get("file_name","?")
        print(f"[{i}] score={h.score:.4f} source={src}")
        print(h.node.text[:280].replace("\n"," "), "...\n")
    return hits

hits2 = debug_retrieve2("What affects speedup in speculative decoding?", k=10)

[1] score=0.7244 source=naacl2025_decoding_speculative_decoding.pdf
Given the promised benefits of speculative de- coding, this paper first focuses on understanding the key factors that dictate the throughput improve- ments that can be obtained. We perform a com- prehensive benchmarking study and profile spec- ulative decoding to characterize bot ...

[2] score=0.6713 source=naacl2025_decoding_speculative_decoding.pdf
Proceedings of the 2025 Conference of the Nations of the Americas Chapter of the Association for Computational Linguistics: Human Language Technologies (V olume 1: Long Papers), pages 6460–6473 April 29 - May 4, 2025 ©2025 Association for Computational Linguistics Decoding Specul ...

[3] score=0.6702 source=naacl2025_decoding_speculative_decoding.pdf
During the autore- gressive decoding, the model generates new text sequentially, a token at a time, building upon the context provided in the prefill phase. Due to its se- quential nature, the autoregressive decoding phase i

In [37]:
print(rag_answer_safe_filtered(
    "Summarize the paper's key findings on what drives speedup in speculative decoding, and common pitfalls.",
    hits2
))

We perform a comprehensive benchmarking study and profile speculative decoding to characterize bottlenecks.


In [38]:
KEY_PHRASES_2 = [
    "speculative decoding", "draft model", "verification", "acceptance rate",
    "speedup", "throughput", "latency", "tokens per second", "kv cache",
    "batch", "prefill", "decode", "compute", "memory", "overhead"
]

def boost_llm_chunks(hits):
    boosted = []
    for h in hits:
        t = h.node.text.lower()
        bonus = sum(1 for k in KEY_PHRASES_2 if k in t)
        boosted.append((h, bonus))
    boosted.sort(key=lambda x: (x[1], x[0].score), reverse=True)
    return [h for h,_ in boosted]

In [39]:
hits2 = boost_llm_chunks(retriever2.retrieve("What affects speedup in speculative decoding?"))
print(rag_answer_safe_filtered(
    "Explain what factors most affect speculative decoding speedup, with evidence from the paper.",
    hits2
))

the latency of the draft model


RAG Routing


In [40]:
RAG_REGISTRY = {
    "health": {
        "label": "WHO Hypertension RAG",
        "retriever": retriever,
        "booster": boost_recommendation_chunks,
    },
    "llm": {
        "label": "Speculative Decoding (NAACL 2025) RAG",
        "retriever": retriever2,
        "booster": boost_llm_chunks,
    }
}

In [41]:
from sentence_transformers import SentenceTransformer, util
import torch

router_encoder = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

ROUTE_EXAMPLES = {
    "health": [
        "WHO guideline pharmacological treatment of hypertension in adults",
        "target blood pressure <140/90 <130 mmHg recommendation",
        "when to start antihypertensive treatment threshold",
        "first-line drug classes for hypertension guideline",
        "strong recommendation moderate-certainty evidence blood pressure"
    ],
    "llm": [
        "speculative decoding speedup depends on acceptance rate",
        "draft model verification overhead in speculative decoding",
        "throughput tokens per second improvement speculative decoding",
        "prefill vs decode bottleneck and KV cache in speculative decoding",
        "how to choose draft model size for speculative decoding"
    ]
}

# Precompute embeddings once
_route_keys = list(ROUTE_EXAMPLES.keys())
_route_texts = [t for k in _route_keys for t in ROUTE_EXAMPLES[k]]
_route_owner = [k for k in _route_keys for _ in ROUTE_EXAMPLES[k]]
_route_emb = router_encoder.encode(_route_texts, normalize_embeddings=True, convert_to_tensor=True)

def route_query(query: str, sim_threshold=0.35, margin=0.03):
    q_emb = router_encoder.encode([query], normalize_embeddings=True, convert_to_tensor=True)
    sims = util.cos_sim(q_emb, _route_emb)[0]  # shape: (num_examples,)

    best_idx = int(torch.argmax(sims).item())
    best_sim = float(sims[best_idx].item())
    best_rag = _route_owner[best_idx]

    # margin vs 2nd best
    top2 = torch.topk(sims, k=min(2, sims.numel())).values
    second_sim = float(top2[1].item()) if top2.numel() > 1 else -1.0
    confident = (best_sim >= sim_threshold) and ((best_sim - second_sim) >= margin)

    return best_rag, {"best_sim": best_sim, "second_sim": second_sim, "confident": confident}

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [42]:
def run_rag(rag_key: str, question: str, top_k=40, keep_k=10):
    rag = RAG_REGISTRY[rag_key]

    hits = rag["retriever"].retrieve(question) if top_k is None else \
           rag["retriever"].retrieve(question)  # retriever already has its own top_k
    # If you want strict top_k control:
    hits = rag["retriever"]._index.as_retriever(similarity_top_k=top_k).retrieve(question) \
           if hasattr(rag["retriever"], "_index") else hits

    # filter citations-like chunks
    useful = [h for h in hits if not looks_like_citations(h.node.text)]
    if not useful:
        useful = hits

    # boost domain-relevant chunks
    boosted = rag["booster"](useful) if rag.get("booster") else useful
    boosted = boosted[:keep_k]

    # answer
    answer = rag_answer_safe_filtered(question, boosted)

    # meta for choosing between multiple rags
    top_score = float(boosted[0].score) if boosted else -1.0
    sources = []
    for h in boosted:
        src = h.node.metadata.get("file_name") or h.node.metadata.get("source") or "unknown"
        sources.append(src)
    sources = list(dict.fromkeys(sources))  # unique, preserve order

    return {
        "rag_key": rag_key,
        "rag_label": rag["label"],
        "answer": answer,
        "top_score": top_score,
        "sources": sources,
    }

In [43]:
def run_rag(rag_key: str, question: str, top_k=40, keep_k=10):
    rag = RAG_REGISTRY[rag_key]

    hits = rag["retriever"].retrieve(question) if top_k is None else \
           rag["retriever"].retrieve(question)  # retriever already has its own top_k
    # If you want strict top_k control:
    hits = rag["retriever"]._index.as_retriever(similarity_top_k=top_k).retrieve(question) \
           if hasattr(rag["retriever"], "_index") else hits

    # filter citations-like chunks
    useful = [h for h in hits if not looks_like_citations(h.node.text)]
    if not useful:
        useful = hits

    # boost domain-relevant chunks
    boosted = rag["booster"](useful) if rag.get("booster") else useful
    boosted = boosted[:keep_k]

    # answer
    answer = rag_answer_safe_filtered(question, boosted)

    # meta for choosing between multiple rags
    top_score = float(boosted[0].score) if boosted else -1.0
    sources = []
    for h in boosted:
        src = h.node.metadata.get("file_name") or h.node.metadata.get("source") or "unknown"
        sources.append(src)
    sources = list(dict.fromkeys(sources))  # unique, preserve order

    return {
        "rag_key": rag_key,
        "rag_label": rag["label"],
        "answer": answer,
        "top_score": top_score,
        "sources": sources,
    }

In [44]:
def looks_like_refusal(ans: str) -> bool:
    a = ans.lower()
    return ("don't know" in a) or ("not enough information" in a) or ("insufficient" in a)

def route_and_answer(question: str, sim_threshold=0.35, margin=0.03):
    choice, info = route_query(question, sim_threshold=sim_threshold, margin=margin)

    # If confident: run only chosen RAG
    if info["confident"]:
        out = run_rag(choice, question)
        out["routing"] = info
        return out

    # Otherwise: run both (or top-2 later when you have 100s)
    out_a = run_rag("health", question)
    out_b = run_rag("llm", question)

    # Pick best: prefer non-refusal, then higher top_score
    a_bad = looks_like_refusal(out_a["answer"])
    b_bad = looks_like_refusal(out_b["answer"])

    if a_bad and not b_bad:
        winner = out_b
    elif b_bad and not a_bad:
        winner = out_a
    else:
        winner = out_a if out_a["top_score"] >= out_b["top_score"] else out_b

    winner["routing"] = info
    winner["fallback_candidates"] = [
        {"rag": out_a["rag_label"], "top_score": out_a["top_score"], "sources": out_a["sources"]},
        {"rag": out_b["rag_label"], "top_score": out_b["top_score"], "sources": out_b["sources"]},
    ]
    return winner

In [45]:
tests = [
    "Summarize WHO recommendations for pharmacological treatment of hypertension in adults.",
    "What target blood pressure does WHO recommend for patients with hypertension and known CVD?",
    "What factors most affect speedup in speculative decoding according to the NAACL 2025 paper?",
    "Explain the role of draft model acceptance rate in speculative decoding speedup.",
]

for q in tests:
    out = route_and_answer(q)
    print("="*90)
    print("Q:", q)
    print("Chosen:", out["rag_label"])
    print("Routing info:", out["routing"])
    print("Sources:", out["sources"])
    print("\nA:\n", out["answer"][:1200])

Token indices sequence length is longer than the specified maximum sequence length for this model (517 > 512). Running this sequence through the model will result in indexing errors


Q: Summarize WHO recommendations for pharmacological treatment of hypertension in adults.
Chosen: WHO Hypertension RAG
Routing info: {'best_sim': 0.9536210298538208, 'second_sim': 0.6302796006202698, 'confident': True}
Sources: ['WHO_Pharmacological treatment of hypertension in adults.pdf']

A:
 Target blood pressure
Q: What target blood pressure does WHO recommend for patients with hypertension and known CVD?
Chosen: WHO Hypertension RAG
Routing info: {'best_sim': 0.6448448896408081, 'second_sim': 0.6383209228515625, 'confident': False}
Sources: ['WHO_Pharmacological treatment of hypertension in adults.pdf']

A:
 130 mmHg
Q: What factors most affect speedup in speculative decoding according to the NAACL 2025 paper?
Chosen: Speculative Decoding (NAACL 2025) RAG
Routing info: {'best_sim': 0.7786735892295837, 'second_sim': 0.7011467218399048, 'confident': True}
Sources: ['naacl2025_decoding_speculative_decoding.pdf']

A:
 latency of the draft model
Q: Explain the role of draft model acce